# YouTube Title Predictor

**Problem.** Small travel channels like my channel, KarmicSteps (~340 subscribers), often have
2–3 title ideas for a video and no data on which will perform better. This project predicts,
from information known at upload, whether a long-form travel video will **beat its own
channel's recent typical views**. The model is used to **rank candidate titles** before publishing.

**Data.** Videos from small travel channels (1k–100k subscribers) collected with the
YouTube Data API on 2026-10-26 (`collect_data.py`, channel selection in `explore_api.ipynb`).

**Target.** `beat_baseline = 1` if a video's views exceed the median of its channel's
previous 20 long-form videos.

**Metric.** ROC AUC, with cross-validation grouped by channel (the model is always scored on
channels it has never seen, which is the situation when scoring KarmicSteps titles).


## 1. Data cleaning

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### 1.1 Load the data and fix the types
`COLLECTION_DATE` is fixed, so the age filter gives the same result on every run.
Empty tags and descriptions are read as NaN, so they're filled with empty text.

In [50]:
COLLECTION_DATE = pd.Timestamp("2026-10-26", tz="UTC")

videos = pd.read_csv("data/videos.csv")
print(videos.shape)
videos.head()

(21814, 9)


,video_id,channel_id,title,published_at,duration,view_count,tags,description,was_live
0,7uNx8yEasm4,UCYAIbGWcLdohnpRh-YLXzRw,Kuala Lumpur might be Asia’s most convenient city,2026-10-24T11:01:11Z,PT17M6S,20404,kuala lumpur|Kuala Lumpur vlog|Kuala Lumpur Ma...,Get 5% off your own tomtoc bag using code *SAR...,False
1,GkcnSulE3SU,UCYAIbGWcLdohnpRh-YLXzRw,This INCREDIBLE city is only 1 hour from KUALA...,2026-10-21T11:01:01Z,PT29M40S,41837,Seremban Malaysia|Seremban travel|Seremban vlo...,We left Melaka behind and took the local buses...,False
2,Vp-tGcRs2EQ,UCYAIbGWcLdohnpRh-YLXzRw,You need to know this if you're heading to Asi...,2026-10-21T07:31:21Z,PT45S,2867,travel tips|travelling asia|southeast asia|bac...,"👉 Before we dive in, comment SLYNUMBER to get ...",False
3,K9nwiOuEIOw,UCYAIbGWcLdohnpRh-YLXzRw,What's on Asia's most mysterious island? 🤔 (Fu...,2026-10-21T04:09:38Z,PT34S,13408,pulau besar|pulau besar melaka|pulau besar mal...,"Just off the coast of Melaka is Pulau Besar, a...",False
4,sSOKZa5xOCY,UCYAIbGWcLdohnpRh-YLXzRw,European countries could never... (Full video ...,2026-10-20T07:33:27Z,PT30S,14537,melaka|malacca|melaka malaysia|malacca malaysi...,We've been in Melaka for a few days and now it...,False


In [51]:
videos["tags"] = videos["tags"].fillna("")
videos["description"] = videos["description"].fillna("")
videos["published_at"] = pd.to_datetime(videos["published_at"])
videos["duration_s"] = videos["duration"].apply(lambda d: pd.Timedelta(d).total_seconds())
videos["age_days"] = (COLLECTION_DATE - videos["published_at"]).dt.days
videos.dtypes

video_id                        str
channel_id                      str
title                           str
published_at    datetime64[us, UTC]
duration                        str
view_count                    int64
tags                            str
description                     str
was_live                       bool
duration_s                  float64
age_days                      int64
dtype: object

### 1.2 Long-form only, no live streams
Shorts can be up to 3 minutes, so only videos longer than 180 s are kept. Past live streams
are removed too: their views behave differently from normal uploads.

In [52]:
videos = videos[videos['duration_s'] > 180]
print("After filtering long-form videos:", len(videos))
videos = videos[~videos["was_live"]]
print("After removing live streams:", len(videos))

After filtering long-form videos: 10890
After removing live streams: 10047


## 2. Target: beat the channel's recent median
Each video is compared with the median views of its own channel's previous 20 long-form
videos. This removes the effect of channel size and uses only information available at upload.

### 2.1 Rolling median of the previous 20 videos
`shift(1)` makes sure a video's own views are never part of its baseline (no leakage).
A channel's first 20 videos have no full baseline (`NaN`), so they're dropped in 2.2.

In [53]:
videos = videos.sort_values(["channel_id", "published_at"])         # channel first, then publish date

videos["baseline"] = (
    videos.groupby("channel_id")["view_count"]
    .transform(lambda s: s.shift(1).rolling(20, min_periods=20).median())
)

videos["beat_baseline"] = (videos["view_count"] > videos["baseline"]).astype(int)

print("Rows with a baseline:", videos["baseline"].notna().sum())
videos[["channel_id", "published_at", "view_count", "baseline", "beat_baseline"]].head(25)


Rows with a baseline: 8696


,channel_id,published_at,view_count,baseline,beat_baseline
16372,UC-9wCcxlOzrejf8ibN5blPg,2022-04-26 12:00:37+00:00,200,NaN,0
16371,UC-9wCcxlOzrejf8ibN5blPg,2022-05-26 11:00:04+00:00,249,NaN,0
16369,UC-9wCcxlOzrejf8ibN5blPg,2022-06-16 05:00:07+00:00,39528,NaN,0
16368,UC-9wCcxlOzrejf8ibN5blPg,2022-06-23 05:00:07+00:00,2077,NaN,0
16367,UC-9wCcxlOzrejf8ibN5blPg,2022-06-30 05:00:19+00:00,4754,NaN,0
16366,UC-9wCcxlOzrejf8ibN5blPg,2022-07-07 05:00:09+00:00,5565,NaN,0
16365,UC-9wCcxlOzrejf8ibN5blPg,2022-07-15 05:16:48+00:00,18524,NaN,0
16364,UC-9wCcxlOzrejf8ibN5blPg,2022-07-22 05:29:14+00:00,4725,NaN,0
16363,UC-9wCcxlOzrejf8ibN5blPg,2022-07-26 05:33:48+00:00,996,NaN,0
16362,UC-9wCcxlOzrejf8ibN5blPg,2022-07-29 05:25:50+00:00,89089,NaN,0


### 2.2 Drop rows without a baseline, apply the age window
Only videos aged **90 days to 3 years** are used as examples.

- The minimum was raised from 30 to 90 days after EDA (section 3.3): videos aged 30–90 days beat their baseline 65% of the time (median 1.4× the baseline), while every older age band is close to 50%. This is most likely a selection effect: the channels were found through search in October 2026, which favours channels that are growing right now.
- Videos older than 3 years may reflect what worked years ago.

Older videos are still used as *history* in the baselines, which is why this filter comes after 2.1.

In [ ]:
df = videos[videos["baseline"].notna()]
print("With a baseline:", len(df))

df = df[(df["age_days"] >= 90) & (df["age_days"] <= (3 * 365))]
print("With age between 90 days and 3 years:", len(df))

print("Channels:", df["channel_id"].nunique())
df["beat_baseline"].value_counts(normalize=True)

In [55]:
per_channel = df["channel_id"].value_counts()
print(per_channel.describe())
print("Channels with < 20 videos:", (per_channel < 20).sum())

count     54.000000
mean      64.425926
std       69.331743
min        1.000000
25%       12.000000
50%       40.000000
75%       93.250000
max      272.000000
Name: count, dtype: float64
Channels with < 20 videos: 19


### 2.3 Language filter and minimum videos per channel

In [56]:
def ascii_share(title):
    letters = [c for c in title if c.isalpha()]
    if not letters:
        return 1.0
    return sum(c.isascii() for c in letters) / len(letters)

df["ascii_share"] = df["title"].apply(ascii_share)
print(df[df["ascii_share"] < 0.9]["title"].head())

1544    Uttarakhand के पहाड़ों पर बसने का सपना ?😍| Rani...
1536    आदि कैलाश ओम पर्वत यात्रा 2025 || DANGEROUS RO...
1529    भारत से KAILASH MANSAROVAR दर्शन​ || Lipulekh ...
1528    Kailash Mansarovar के बाद NEPAL पहुचँ गए ?? ||...
1527    Kailash के बाद की पहली सुबह Dharchula में ? 😳|...
Name: title, dtype: str


Channels where more than 10% of titles are in a non-Latin script (Hindi, Punjabi, Tamil, Kannada, Korean) are dropped entirely, keeping channels that publish mainly in English, like KarmicSteps. Remaining individual non-English titles are dropped too. Known limitation: mixed Hindi–English titles and Latin-script languages (e.g. Turkish) can still slip through.

In [57]:
# 1. Channel-level: share of each channel's titles that look non-English
df["non_english"] = df["ascii_share"] < 0.9
channel_share = df.groupby("channel_id")["non_english"].mean()
english_channels = channel_share[channel_share <= 0.1].index
df = df[df["channel_id"].isin(english_channels)]
print("After dropping non-English channels:", len(df), "videos,", df["channel_id"].nunique(), "channels")

# 2. Title-level: drop remaining non-English titles
df = df[~df["non_english"]]
print("After dropping non-English titles:", len(df))

# 3. Per-channel floor
counts = df["channel_id"].value_counts()
keep = counts[counts >= 20].index
df = df[df["channel_id"].isin(keep)]
print("After the 20-video floor:", len(df), "videos,", df["channel_id"].nunique(), "channels")

df["beat_baseline"].value_counts(normalize=True)


After dropping non-English channels: 2913 videos, 46 channels
After dropping non-English titles: 2910
After the 20-video floor: 2773 videos, 29 channels


beat_baseline
0    0.504147
1    0.495853
Name: proportion, dtype: float64

### 2.4 Save the clean dataset
`train.py` loads this file, so the cleaning steps don't have to be repeated.

In [58]:
df.to_csv("data/videos_clean.csv", index=False)

## 3. Exploratory data analysis

### 3.1 Target balance

In [ ]:
plt.figure(figsize=(8, 6))
df['beat_baseline'].value_counts().sort_index().plot(kind='bar')
plt.xticks([0, 1], ['Below baseline (0)', 'Beat baseline (1)'], rotation=0)
plt.ylabel('Videos')
plt.title('Target balance: beat the channel baseline?')
plt.show()

### 3.2 Videos per channel
The 29 channels contribute between 22 and 272 videos each. The four largest channels hold about 35% of all videos, so they have more weight in training, but no single channel dominates.

In [ ]:
channel_names = pd.read_csv("data/channels.csv").set_index("channel_id")["title"]
per_channel = df["channel_id"].map(channel_names).value_counts()

per_channel.plot(kind="bar", figsize=(12, 4), title="Videos per channel")
plt.ylabel("Videos")
plt.show()

### 3.3 Does the target depend on video age?
With the original 30-day minimum, the newest videos behaved differently from the rest:

| Age band | Videos | Beat baseline | Median views ÷ baseline |
|---|---|---|---|
| 30–90 days | 192 | 65% | 1.40× |
| 90–180 days | 285 | 47% | 0.98× |
| 180–365 days | 604 | 50% | 1.00× |
| 1–2 years | 996 | 49% | 0.98× |
| 2–3 years | 885 | 51% | 1.01× |

So the minimum age was raised to 90 days (see 2.2). After the change, every age band is close to 50%:

In [62]:
df["age_band"] = pd.cut(df["age_days"], bins=[90, 180, 365, 730, 1095])
df.groupby("age_band")['beat_baseline'].mean()

age_band
(90, 180]      0.473684
(180, 365]     0.500000
(365, 730]     0.488956
(730, 1095]    0.507345
Name: beat_baseline, dtype: float64

### 3.4 Numeric features (duration, publish time, tags, description length)

### 3.5 Title words: most common words for beat vs not beat